# Week 11: Serving Turns a Fitted Model into a Service

This notebook follows the reviewed Week 11 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Serving Turns a Fitted Model into a Service
2. HTTP Defines Requests and Responses
3. JSON and Validation Define the API Contract
4. FastAPI Connects a Typed Contract to Inference
5. Health Checks Separate Availability from Prediction Quality
6. Docker Packages the Application Environment
7. Configuration and Secrets Belong Outside Source Code
8. EC2 Is a Virtual Server You Must Operate
9. Security Groups Expose Only Required Traffic
10. Logs and Metrics Make Failures Observable
11. Cost Controls Start Before the Instance
12. Guided Lab: Deploy, Verify, and Clean Up

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Serving Turns a Fitted Model into a Service

**Model serving** makes a fitted model available to another program.

A serving request contains feature values. The service:

1. validates the request;
2. applies the saved preprocessing pipeline;
3. calculates inference;
4. formats a response;
5. records operational evidence.

Training and serving must use the same feature definitions and fitted artifact. A successful HTTP response does not prove the prediction is correct.

### Work it out first

Request: `{"age": 35, "city": "Kandy"}`

The service validates types, applies the saved imputer and encoder, returns a probability and model version, then logs latency without exposing secrets.

### Notebook bridge

This week wraps the Week 10 pipeline in an API rather than retraining it inside each request.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
features = {"age": 35, "city": "Kandy"}
prediction = pipeline.predict_proba([features])[0, 1]
print(prediction)

Expected output:

```text
A positive-class probability produced by the saved pipeline.
```


## 2. HTTP Defines Requests and Responses

**HTTP** is a protocol for exchanging requests and responses.

A request contains:

- method such as `GET` or `POST`;
- path such as `/predict`;
- headers such as content type or authorization;
- optional body containing input data.

A response contains a status code, headers, and optional body.

Common codes:

- `200`: success
- `400` or `422`: invalid client input
- `401` or `403`: authentication or permission failure
- `500`: unexpected server failure

### Work it out first

`POST /predict` sends feature JSON because prediction creates a computation from a structured body.

Valid input returns `200`. Missing required age returns a validation error, commonly `422` in FastAPI.

### Notebook bridge

Learners will send test requests to the local FastAPI application.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
POST /predict HTTP/1.1
Content-Type: application/json

{"age": 35, "city": "Kandy"}

Expected output:

```text
HTTP 200 with a structured JSON prediction response.
```


## 3. JSON and Validation Define the API Contract

**JSON** represents objects, arrays, strings, numbers, booleans, and null.

An **API contract** states required fields, data types, ranges, response fields, and error behaviour.

Validation should reject:

- missing required fields;
- wrong types;
- values outside valid ranges;
- unknown fields when strict input is required;
- requests that exceed size limits.

Validation protects the service boundary; it does not replace data-drift monitoring.

### Work it out first

Contract:

- `age`: integer from `18` to `100`
- `city`: non-empty string

`{"age": "thirty-five"}` fails type and missing-city checks before model inference.

### Notebook bridge

The request schema must match the columns expected by the persisted pipeline.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from pydantic import BaseModel, Field

class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=100)
    city: str = Field(min_length=1)

Expected output:

```text
Valid JSON becomes a typed request object; invalid JSON returns field-level errors.
```


## 4. FastAPI Connects a Typed Contract to Inference

**FastAPI** maps HTTP paths and methods to Python functions.

The endpoint should:

1. accept a typed request;
2. convert it into the model's row format;
3. call the loaded pipeline;
4. return JSON-safe values;
5. convert expected failures into clear client errors.

Load the model once when the application starts, not once per request.

### Work it out first

For probability `0.73` and threshold `0.50`, return:

```json
{"label": 1, "probability": 0.73, "model_version": "2026-07-28"}
```

The version lets logs and clients identify the producing artifact.

### Notebook bridge

The lab uses the complete saved pipeline and a small local API test.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
@app.post("/predict")
def predict(request: PredictionRequest):
    row = pd.DataFrame([request.model_dump()])
    probability = float(model.predict_proba(row)[0, 1])
    return {"label": int(probability >= 0.5), "probability": probability}

Expected output:

```text
A validated JSON response containing a class label and probability.
```


## 5. Health Checks Separate Availability from Prediction Quality

A **health check** is a lightweight endpoint used by operators or infrastructure.

- **Liveness:** is the process running?
- **Readiness:** can it currently serve valid requests?
- **Startup:** has initialization completed?

A readiness check may confirm the model loaded and required dependencies are available. It should not perform an expensive full prediction on every probe.

Health checks do not measure model accuracy or data drift.

### Work it out first

`GET /health/ready`

Ready response:

```json
{"status": "ready", "model_version": "2026-07-28"}
```

If the model failed to load, return a non-success status so traffic is not routed to the process.

### Notebook bridge

The deployment lab verifies health before sending prediction traffic.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
@app.get("/health/ready")
def ready():
    return {"status": "ready", "model_version": MODEL_VERSION}

Expected output:

```text
HTTP 200 only after the application artifact is ready for inference.
```


## 6. Docker Packages the Application Environment

A Docker **image** is an immutable filesystem and configuration template. A **container** is a running process created from an image.

A model API image should include:

- application code;
- pinned dependencies;
- trusted model artifact;
- startup command;
- non-secret defaults.

Build once and run the same image in test and production. Use a small trusted base image, a non-root user, and explicit version tags or digests.

### Work it out first

Build:

`docker build -t dsacademy-model:1.0 .`

Run:

`docker run -p 8000:8000 dsacademy-model:1.0`

The host port `8000` forwards to the container service port.

### Notebook bridge

The lab containerizes the API only after local validation passes.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["fastapi", "run", "app.py", "--port", "8000"]

Expected output:

```text
One reproducible image that starts the API on port 8000.
```


## 7. Configuration and Secrets Belong Outside Source Code

**Configuration** changes application behaviour between environments. A **secret** grants access, such as an API key or password.

Environment variables can supply configuration at runtime:

`MODEL_PATH`, `LOG_LEVEL`, `PREDICTION_THRESHOLD`

Do not commit `.env` files containing real credentials. In production, use a managed secret store or scoped instance role. Rotate exposed credentials and avoid printing them in logs.

### Work it out first

Unsafe:

```python
AWS_SECRET = "actual-secret"
```

Safer runtime lookup:

```python
threshold = float(os.environ["PREDICTION_THRESHOLD"])
```

The application fails clearly if required configuration is missing.

### Notebook bridge

The deployment exercise keeps environment-specific values outside the application module.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import os

MODEL_PATH = os.environ.get("MODEL_PATH", "/app/model.joblib")

Expected output:

```text
The runtime value is used when set; otherwise the non-secret default path is used.
```


## 8. EC2 Is a Virtual Server You Must Operate

An Amazon EC2 **instance** is a virtual server.

Key components:

- AMI: operating-system image;
- instance type: CPU and memory capacity;
- EBS volume: persistent block storage;
- VPC and subnet: network placement;
- security group: allowed network traffic;
- key pair or managed access method;
- IAM role: scoped AWS permissions.

A low-cost instance can run for months, but you remain responsible for patches, availability, backups, security, monitoring, and charges while it runs.

### Work it out first

For a small CPU-only API, choose the smallest architecture-compatible instance that meets measured memory needs. Start with one instance, no load balancer, and a documented restart procedure.

### Notebook bridge

The lab can deploy to EC2 after local Docker verification; no GPU is required for the small model.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
AMI -> EC2 instance -> Docker container -> FastAPI -> model pipeline

Expected output:

```text
One reachable inference service with an explicit artifact and network path.
```


## 9. Security Groups Expose Only Required Traffic

An EC2 **security group** is a stateful virtual firewall.

Inbound rules should allow only required ports and sources:

- SSH `22`: restrict to an administrator IP, or avoid public SSH by using managed access;
- HTTP `80`: redirect to HTTPS when used;
- HTTPS `443`: public only when the API is intended to be public;
- application port `8000`: keep private behind a reverse proxy when possible.

Never expose every port to `0.0.0.0/0`.

### Work it out first

Safer rules:

- `443` from intended clients;
- `22` from one administrator CIDR;
- no public rule for `8000`.

The reverse proxy terminates TLS and forwards internally to FastAPI.

### Notebook bridge

The deployment checklist records every open port, protocol, source, and reason.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
Internet :443 -> reverse proxy -> localhost:8000 -> FastAPI
Admin IP :22 -> EC2

Expected output:

```text
Only encrypted client traffic and restricted administration reach the instance.
```


## 10. Logs and Metrics Make Failures Observable

**Logs** record discrete events. **Metrics** summarize numerical behaviour over time.

Record:

- timestamp and request ID;
- model version;
- status code and error type;
- latency;
- request and prediction counts;
- resource use;
- input-schema failures;
- carefully designed drift indicators.

Do not log secrets or unnecessary personal data. Prediction quality often arrives later, when actual outcomes become available.

### Work it out first

From `1,000` requests:

- `990` successful;
- `8` validation failures;
- `2` server failures.

Server error rate:

`2 / 1000 = 0.002 = 0.2%`

### Notebook bridge

The lab verifies logs for success, validation failure, and unexpected failure.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
logger.info(
    "prediction_complete",
    extra={"request_id": request_id, "latency_ms": 42, "model_version": MODEL_VERSION},
)

Expected output:

```text
A structured event that can be searched and aggregated without exposing features.
```


## 11. Cost Controls Start Before the Instance

Cloud costs continue while resources run.

Before deployment:

- confirm current Free Tier or credit eligibility;
- estimate instance and storage cost;
- choose the smallest measured capacity;
- set AWS Budgets alerts;
- tag resources with owner and purpose;
- define stop or termination dates;
- remove unused snapshots, IPs, and volumes;
- avoid optional paid services unless justified.

Stopping an instance may leave EBS storage billed. Termination and deletion policies must be deliberate.

### Work it out first

Monthly estimate:

`hourly rate x 24 x 30 + storage + data transfer`

Even when compute is covered by credits, storage or transfer may not be. The exact rate must be checked for region, account, and date.

### Notebook bridge

The deployment lab includes a resource inventory and cleanup verification.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
Budget alert: 50% -> review
Budget alert: 80% -> stop nonessential service
Budget alert: 100% -> terminate lab resources

Expected output:

```text
A documented response before spending exceeds the agreed limit.
```


## 12. Guided Lab: Deploy, Verify, and Clean Up

Complete:

1. local API contract tests;
2. model loaded once at startup;
3. health and prediction endpoints;
4. Docker image build and local run;
5. environment-based configuration;
6. smallest suitable EC2 instance;
7. restricted security-group rules;
8. HTTPS or a clearly bounded private test;
9. success and failure logs;
10. budget alert and resource tags;
11. rollback procedure;
12. stop, terminate, or cleanup verification.

### Work it out first

Verification sequence:

`GET /health/ready -> 200`  
valid `POST /predict -> 200`  
invalid input `-> 422`  
unknown path `-> 404`  
logs contain request IDs and no secrets.

### Notebook bridge

This deployment lab packages the Week 10 pipeline and prepares learners for model-backed AI services.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
curl -sS https://example.test/health/ready
curl -sS -X POST https://example.test/predict \
  -H 'content-type: application/json' \
  -d '{"age":35,"city":"Kandy"}'

Expected output:

```text
A ready response followed by a typed prediction response.
```


## Guided lab

Complete:

1. local API contract tests;
2. model loaded once at startup;
3. health and prediction endpoints;
4. Docker image build and local run;
5. environment-based configuration;
6. smallest suitable EC2 instance;
7. restricted security-group rules;
8. HTTPS or a clearly bounded private test;
9. success and failure logs;
10. budget alert and resource tags;
11. rollback procedure;
12. stop, terminate, or cleanup verification.

### Reference result

Verification sequence:

`GET /health/ready -> 200`  
valid `POST /predict -> 200`  
invalid input `-> 422`  
unknown path `-> 404`  
logs contain request IDs and no secrets.


In [ ]:
# Guided lab workspace: Week 11
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://scikit-learn.org/stable/model_persistence.html>
- <https://fastapi.tiangolo.com/tutorial/>
- <https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview>
- <https://fastapi.tiangolo.com/tutorial/first-steps/>
- <https://fastapi.tiangolo.com/tutorial/body/>
- <https://docs.pydantic.dev/latest/concepts/fields/>
- <https://fastapi.tiangolo.com/tutorial/path-operation-configuration/>
- <https://fastapi.tiangolo.com/tutorial/handling-errors/>
- <https://kubernetes.io/docs/concepts/configuration/liveness-readiness-startup-probes/>
- <https://fastapi.tiangolo.com/advanced/events/>
- <https://fastapi.tiangolo.com/deployment/docker/>
- <https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/>
- <https://docs.docker.com/compose/how-tos/environment-variables/>
- <https://docs.aws.amazon.com/secretsmanager/latest/userguide/best-practices.html>
- <https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/EC2_GetStarted.html>
- <https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html>
- <https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-security-groups.html>
- <https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/creating-security-group.html>
- <https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/GettingStarted.html>
- <https://opentelemetry.io/docs/concepts/signals/>
- <https://docs.aws.amazon.com/hands-on/latest/control-your-costs-free-tier-budgets/control-your-costs-free-tier-budgets.html>
- <https://docs.aws.amazon.com/awsaccountbilling/latest/aboutv2/free-tier.html>
- <https://fastapi.tiangolo.com/deployment/>